# Storage features
Revised historical AGSI proxy, assumed available two calendar days after each gas day. Seasonal normals use prior years only. Availability dates are explicit; original publication vintages are unavailable.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("TTF_PROJECT_ROOT", Path.cwd().parent))
RAW = ROOT / "1. Raw Data"
OUT = ROOT / "2. Data Processing"
funda = pd.read_csv(RAW / "funda.csv", parse_dates=["gasDayStart"])
funda = funda.sort_values("gasDayStart").reset_index(drop=True)

In [2]:
funda["agsi_estimated"] = funda["status"].eq("E").astype(int)
feb29 = funda["gasDayStart"].dt.month.eq(2) & funda["gasDayStart"].dt.day.eq(29)
key = funda["gasDayStart"].dt.strftime("%m-%d").mask(feb29, "02-28")
year = funda["gasDayStart"].dt.year
hist = funda.loc[~feb29, ["gasDayStart", "full"]].copy()
hist["key"] = hist["gasDayStart"].dt.strftime("%m-%d")
hist["year"] = hist["gasDayStart"].dt.year
lookup = hist.set_index(["key", "year"])["full"]
prior = pd.DataFrame({lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup) for lag in range(1, 6)})
funda["full_norm5y"] = prior.mean(axis=1, skipna=False)
funda["storage_dev"] = funda["full"] - funda["full_norm5y"]
funda["storage_dev_sq"] = funda["storage_dev"] ** 2

In [3]:
funda["net_inj"] = -funda["netWithdrawal"] / 1000
funda["inj_util"] = funda["injection"] / funda["injectionCapacity"].replace(0, np.nan)
funda["wd_util"] = funda["withdrawal"] / funda["withdrawalCapacity"].replace(0, np.nan)
funda = funda.drop(columns=["trend", "consumption", "consumptionFull", "status", "gasInStorage", "workingGasVolume", "injection", "withdrawal", "netWithdrawal", "injectionCapacity", "withdrawalCapacity"])
funda["storage_gas_day"] = funda["gasDayStart"]
funda["storage_available_date"] = funda["storage_gas_day"] + pd.Timedelta(days=2)
funda["storage_revised_proxy"] = True
funda["gasDayStart"] = funda["storage_available_date"]
funda.to_csv(OUT / "funda_processed.csv", index=False)